# Modul B · Kapitel 4 — Durchsatz und Last

> 🛠️ **Workshop-Version:** Bearbeite die zwei markierten Aufgaben.

**Lernziel:** Du kannst Latenz messen und aus Messwerten eine Kapazitätsgrenze ableiten.

Dieses Notebook folgt einem kurzen Pfad: Begriff verstehen → Rechnung oder
Messung durchführen → Ergebnis für eine Deployment-Entscheidung nutzen.
Programmiert werden nur zwei Kernstellen: eine Messreihe und Kapazitätsplanung. Hilfs- und
Visualisierungscode ist bewusst vorgegeben.


## 0 · Setup

Die nächsten Zellen laden Messfunktionen, Fülltexte und Planungsszenarien. Für Live-Messungen muss der konfigurierte Modellserver laufen.


In [ ]:
import sys
from pathlib import Path

# helfer.py liegt neben dem Notebook. Der Suchlauf findet es auch, wenn das
# Arbeitsverzeichnis woanders liegt — etwa in Colab.
for kandidat in [Path.cwd(), Path.cwd() / "04_deployment", *Path.cwd().parents]:
    if (kandidat / "helfer.py").exists():
        sys.path.insert(0, str(kandidat))
        break

try:
    import matplotlib
    import pandas
except ImportError:
    %pip install -q matplotlib pandas openai tiktoken
    import matplotlib

import json
import math
import os
import statistics
import threading
import time
import urllib.request
import uuid
from concurrent.futures import ThreadPoolExecutor

import matplotlib.pyplot as plt
import numpy as np

import helfer
from helfer import GB, MODELL, lade_daten, messe_anfrage, zeige_tabelle

# Die nativen Ollama-Endpunkte liegen neben der OpenAI-Naht, nicht unter /v1.
OLLAMA = helfer.BASIS_URL.replace("/v1", "")

# Diese Maschine. Auf einem anderen Rechner wird hier der passende Eintrag aus
# daten/hardware.json gewählt — Abschnitt 6 rechnet mit der Bandbreite.
HARDWARE_HIER = "MacBook Pro M4 Max (128 GB)"
HIER = {k["name"]: k for k in lade_daten("hardware")["karten"]}[HARDWARE_HIER]

# Einheitliche Farben für alle Diagramme in diesem Notebook
BLAU, ORANGE, TEAL, GRAU = "#2563eb", "#e8590c", "#0d9488", "#6b7280"

plt.rcParams.update({
    "figure.figsize": (9, 4.5),
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": GRAU,
    "axes.grid": True,
    "axes.axisbelow": True,
    "grid.color": "#e5e7eb",
    "grid.linewidth": 0.8,
    "axes.titlesize": 13,
    "axes.titleweight": "bold",
    "font.size": 10,
})

print(f"Modell:     {MODELL} über {helfer.BASIS_URL}")
print(f"Hardware:   {HIER['name']}, {HIER['vram_gb']} GB, "
      f"{HIER['speicherbandbreite_gb_s']} GB/s Speicherbandbreite")
print("Setup fertig ✔")


In [ ]:
# ▶️ Fülltext, Planungssituationen, Prompts — und der Server
LAST_DATEI = lade_daten("04_lasttest")
LOGZEILEN = LAST_DATEI["logzeilen"]
SITUATIONEN = LAST_DATEI["situationen"]

CVE_TEXT = (
    "CVE-2026-3224 — CVSS 9.1. An unauthenticated attacker can send a crafted SAML "
    "assertion to the /sso/acs endpoint of NorthGate VPN Gateway 7.2 to 7.4 and obtain a "
    "valid administrator session. A public proof of concept exists. Patch 7.4.3 is "
    "available. Affected in our estate: 14 gateways, 3 of them reachable from the "
    "internet."
)

# Der Prompt für alle Lastmessungen. Kurz, damit das Prefill nicht ins Gewicht
# fällt, und mit fester Antwortlänge, damit jede Anfrage dieselbe Arbeit ist.
LAST_PROMPT = f"{CVE_TEXT}\n\nWrite a two sentence briefing for a CISO."
LAST_TOKENS = 80


def nativ(pfad, nutzlast=None):
    """Ruft einen nativen Ollama-Endpunkt auf und gibt das JSON zurück."""
    daten = json.dumps(nutzlast).encode() if nutzlast else None
    anfrage = urllib.request.Request(f"{OLLAMA}{pfad}", data=daten,
                                     headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(anfrage, timeout=20) as antwort:
        return json.load(antwort)


print(f"{len(LOGZEILEN)} Logzeilen als Fülltext, {len(SITUATIONEN)} Planungssituationen "
      f"(Stand {LAST_DATEI['stand']})")
print()

try:
    print(f"Ollama {nativ('/api/version')['version']}, installiert: "
          + ", ".join(sorted(m["name"] for m in nativ("/api/tags")["models"])))
except Exception as fehler:
    print(f"Ollama nicht erreichbar ({type(fehler).__name__}) — dieses Notebook braucht es.")

# Der erste Aufruf lädt das Modell in den Speicher. Er gehört in keine Messung.
aufwaermen = messe_anfrage("Warm up.", max_tokens=8)
print(f"aufgewärmt: {aufwaermen['sekunden']:.2f} s für {aufwaermen['antwort_tokens']} Tokens")


## 1 · Drei verschiedene Größen

TTFT misst die Zeit bis zum ersten Token, Tokens/s die Generationsrate und Durchsatz die Arbeit des Gesamtsystems. Keine Zahl ersetzt die anderen.


In [ ]:
# ▶️ Eine Anfrage, alle drei Größen daran
lauf = messe_anfrage(LAST_PROMPT, max_tokens=LAST_TOKENS)

generierung = lauf["sekunden"] - lauf["ttft"]

zeige_tabelle([
    {"Größe": "Latency", "Wert": f"{lauf['sekunden']:.2f} s",
     "gemessen als": "Start der Anfrage bis zum letzten Token"},
    {"Größe": "TTFT", "Wert": f"{lauf['ttft']:.2f} s",
     "gemessen als": "Start der Anfrage bis zum ersten Token"},
    {"Größe": "Generierung", "Wert": f"{generierung:.2f} s",
     "gemessen als": "Latency − TTFT"},
    {"Größe": "Rate dieser Anfrage", "Wert": f"{lauf['tokens_pro_sekunde']:.0f} Tokens/s",
     "gemessen als": "Antwort-Tokens ÷ Generierungszeit"},
    {"Größe": "Throughput des Servers", "Wert": "—",
     "gemessen als": "geht an einer einzelnen Anfrage nicht — siehe Abschnitt 4"},
])

print(f"{lauf['prompt_tokens']} Prompt-Tokens, {lauf['antwort_tokens']} Antwort-Tokens")
print(f"Anteil der TTFT an der Latency: {lauf['ttft'] / lauf['sekunden']:.0%}")
print()
print(lauf["antwort"][:240], "…")


## 2 · Prefill und Decode

Im Prefill verarbeitet das Modell den gesamten Prompt. Im Decode erzeugt es Token für Token. Lange Prompts erhöhen vor allem die TTFT.


In [ ]:
# ▶️ Der Prompt-Bauer. Die Marke steht vorne, sonst greift der Prefill-Cache.
KURZE_FRAGE = "Answer with one word: is this excerpt relevant to the CVE?"


def baue_prompt(zeilen_anzahl, auftrag=KURZE_FRAGE):
    """Ein Prompt aus `zeilen_anzahl` Logzeilen, jedes Mal mit neuer Marke."""
    auszug = "\n".join(LOGZEILEN[:zeilen_anzahl])
    return (f"Case {uuid.uuid4().hex}\n"
            f"{CVE_TEXT}\n\nLog excerpt:\n{auszug}\n\n{auftrag}")


LAENGEN = [2, 12, 25, 50, 65, 82]      # Zeilen → rund 250 bis 4000 Prompt-Tokens

for n in (LAENGEN[0], LAENGEN[-1]):
    probe = messe_anfrage(baue_prompt(n), max_tokens=8)
    print(f"{n:3d} Zeilen → {probe['prompt_tokens']:5d} Prompt-Tokens, "
          f"TTFT {probe['ttft']:.2f} s")


In [ ]:
# Vorgegebene Hilfsfunktion
def messe_prefill(zeilenzahlen, wiederholungen=3, max_tokens=8):
    """TTFT über wachsende Promptlängen, je Länge der Median mehrerer Läufe."""
    zeilen = []
    for n in zeilenzahlen:
        # Der Prompt muss in der Schleife entstehen: Jeder Lauf braucht eine
        # eigene Marke, sonst misst die zweite Wiederholung den Prefill-Cache.
        laeufe = [messe_anfrage(baue_prompt(n), max_tokens=max_tokens)
                  for _ in range(wiederholungen)]
        ttfts = [l["ttft"] for l in laeufe]
        zeilen.append({
            "zeilen": n,
            "prompt_tokens": laeufe[-1]["prompt_tokens"],
            "ttft": statistics.median(ttfts),
            "ttft_min": min(ttfts),
            "ttft_max": max(ttfts),
        })
    return zeilen


## 3 · Messreihen statt Einzelwerte

Warme Modelle, Hintergrundlast und Ausreißer verzerren Einzelmessungen. Mehrere Läufe und Perzentile zeigen das reale Verhalten besser.


In [ ]:
# ▶️ Perzentil nach der Methode „nächster Rang" — ohne Interpolation
def perzentil(werte, anteil):
    """Der kleinste Messwert, unter oder auf dem `anteil` aller Werte liegen."""
    sortiert = sorted(werte)
    rang = math.ceil(anteil * len(sortiert))
    return sortiert[min(len(sortiert) - 1, max(0, rang - 1))]


probe = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
print(f"p50 von {probe}: {perzentil(probe, 0.50)}")
print(f"p95 von {probe}: {perzentil(probe, 0.95)}")
print("Bei zehn Werten ist p95 der größte. Für belastbare Perzentile braucht es mehr Läufe.")


### 🛠️ Aufgabe 1 — Eine Messreihe auswerten

Führe mehrere Läufe aus und berechne robuste Kennzahlen einschließlich Median und p95.

Führe danach den Selbsttest aus. Die ausgefüllte Variante steht in der Lösungsversion des Notebooks.


In [ ]:
def messreihe(laeufe=20, prompt=LAST_PROMPT, max_tokens=LAST_TOKENS):
    """Dieselbe Anfrage mehrfach, ausgewertet über Perzentile statt über einen Wert."""
    # TODO: Ersetze die nächste Zeile
    raise NotImplementedError("Aufgabe 1: messreihe() implementieren")


In [ ]:
# ✅ Selbsttest
RUHIG = messreihe()

FELDER = {"laeufe", "p50", "p95", "min", "max", "streuung", "ttft_p50", "ttft_p95",
          "tokens", "latenzen"}
assert set(RUHIG) == FELDER, f"Andere Schlüssel als erwartet: {set(RUHIG) ^ FELDER}"
assert len(RUHIG["latenzen"]) == RUHIG["laeufe"] == 20
assert RUHIG["min"] <= RUHIG["p50"] <= RUHIG["p95"] <= RUHIG["max"], \
    "min ≤ p50 ≤ p95 ≤ max — sonst stimmt die Sortierung nicht"
assert RUHIG["ttft_p50"] <= RUHIG["p50"], "Die TTFT steckt in der Latency drin"
assert abs(RUHIG["streuung"] - (RUHIG["max"] - RUHIG["min"]) / RUHIG["p50"]) < 1e-9
assert 0 < RUHIG["tokens"] <= LAST_TOKENS, \
    f"Höchstens {LAST_TOKENS} Antwort-Tokens erwartet, gezählt {RUHIG['tokens']}"

print("✅ Aufgabe 1 gelöst")
print()
print(f"{RUHIG['laeufe']} Läufe, ruhige Maschine, {RUHIG['tokens']} Antwort-Tokens je Anfrage")
print(f"  Latency   p50 {RUHIG['p50']:.2f} s   p95 {RUHIG['p95']:.2f} s   "
      f"min {RUHIG['min']:.2f} s   max {RUHIG['max']:.2f} s")
print(f"  TTFT      p50 {RUHIG['ttft_p50']:.2f} s   p95 {RUHIG['ttft_p95']:.2f} s")
print(f"  Streuung  {RUHIG['streuung']:.0%} des Medians")
print()
print("  einzelne Läufe: " + " ".join(f"{l:.2f}" for l in RUHIG["latenzen"]))


In [ ]:
# ▶️ Dieselbe Messreihe, während ein zweiter Client denselben Server benutzt
schluss = threading.Event()


def stoerer():
    """Ein zweiter Client, der ohne Pause Anfragen stellt."""
    while not schluss.is_set():
        messe_anfrage(LAST_PROMPT, max_tokens=LAST_TOKENS)


faden = threading.Thread(target=stoerer, daemon=True)
faden.start()
time.sleep(1.0)                       # der Störer soll erst richtig laufen

UNTER_LAST = messreihe(laeufe=12)

schluss.set()
faden.join(timeout=60)

zeige_tabelle([
    {"Messreihe": "ruhige Maschine", "Läufe": RUHIG["laeufe"], "p50 (s)": RUHIG["p50"],
     "p95 (s)": RUHIG["p95"], "TTFT p50 (s)": RUHIG["ttft_p50"],
     "Streuung": f"{RUHIG['streuung']:.0%}"},
    {"Messreihe": "ein zweiter Client", "Läufe": UNTER_LAST["laeufe"],
     "p50 (s)": UNTER_LAST["p50"], "p95 (s)": UNTER_LAST["p95"],
     "TTFT p50 (s)": UNTER_LAST["ttft_p50"], "Streuung": f"{UNTER_LAST['streuung']:.0%}"},
])

print(f"Der Median steigt um {UNTER_LAST['p50'] / RUHIG['p50'] - 1:+.0%}, "
      f"die TTFT um {UNTER_LAST['ttft_p50'] / RUHIG['ttft_p50'] - 1:+.0%}.")


## 4 · Last erzeugen

Parallele Anfragen zeigen, ob eine Runtime wirklich gleichzeitig arbeitet oder nur eine Warteschlange aufbaut.


In [ ]:
# Vorgegebene Hilfsfunktion
def last_test(anzahl_parallel, anfragen, prompt=LAST_PROMPT, max_tokens=LAST_TOKENS):
    """`anfragen` Anfragen über `anzahl_parallel` Threads, je Anfrage und über alles gemessen."""
    def eine(_):
        return messe_anfrage(prompt, max_tokens=max_tokens)

    start = time.perf_counter()
    with ThreadPoolExecutor(max_workers=anzahl_parallel) as pool:
        laeufe = list(pool.map(eine, range(anfragen)))
    wanduhr = time.perf_counter() - start

    latenzen = [l["sekunden"] for l in laeufe]
    ttfts = [l["ttft"] for l in laeufe]
    tokens = sum(l["antwort_tokens"] for l in laeufe)

    return {
        "parallel": anzahl_parallel,
        "anfragen": anfragen,
        "wanduhr": wanduhr,
        "p50": perzentil(latenzen, 0.50),
        "p95": perzentil(latenzen, 0.95),
        "ttft_p50": perzentil(ttfts, 0.50),
        "ttft_p95": perzentil(ttfts, 0.95),
        "tokens": tokens,
        "anfragen_pro_sekunde": anfragen / wanduhr,
        "tokens_pro_sekunde": tokens / wanduhr,
    }


## 5 · Den Befund lesen

Steigt die Latenz fast proportional zur Zahl paralleler Anfragen, werden sie weitgehend seriell abgearbeitet.


In [ ]:
# ▶️ Dieselbe Last mit wachsender Parallelität messen
PARALLELITAETEN = [1, 2, 4]
LAST = [last_test(parallel, anfragen=max(4, parallel * 2))
        for parallel in PARALLELITAETEN]
basis = LAST[0]

zeige_tabelle(LAST)


In [ ]:
# ▶️ Was hier gemessen wurde — und was daraus folgt
letzte = LAST[-1]
beschleunigung = letzte["tokens_pro_sekunde"] / basis["tokens_pro_sekunde"]
verlangsamung = letzte["p50"] / basis["p50"]

print(f"Ollama-Version:          {nativ('/api/version')['version']}")
print(f"OLLAMA_NUM_PARALLEL:     "
      f"{os.environ.get('OLLAMA_NUM_PARALLEL', '(im Notebook nicht gesetzt)')}")
print()
print("/api/ps — was gerade geladen ist:")
geladen = nativ("/api/ps")["models"]
if geladen:
    zeige_tabelle([{
        "Modell": m["name"],
        "im Speicher (GB)": m["size"] / GB,
        "davon VRAM (GB)": m["size_vram"] / GB,
        "Kontextfenster": m["context_length"],
    } for m in geladen])
else:
    print("   (gerade nichts)")

print()
print(f"von 1 auf {letzte['parallel']} gleichzeitigen Anfragen:")
print(f"   Gesamtrate   {basis['tokens_pro_sekunde']:6.1f} → "
      f"{letzte['tokens_pro_sekunde']:6.1f} Tokens/s   (×{beschleunigung:.2f})")
print(f"   Latency p50  {basis['p50']:6.2f} → {letzte['p50']:6.2f} s   "
      f"(×{verlangsamung:.2f})")
print(f"   TTFT p50     {basis['ttft_p50']:6.2f} → {letzte['ttft_p50']:6.2f} s   "
      f"(×{letzte['ttft_p50'] / basis['ttft_p50']:.2f})")
print()
if beschleunigung < 1.5:
    print(f"Befund: Der Server serialisiert. Die Gesamtrate ändert sich um "
          f"{beschleunigung - 1:+.0%}, während die Latency um {verlangsamung - 1:+.0%} "
          f"wächst.\nDie Anfragen teilen sich keine Rechenzeit, sie stehen "
          f"hintereinander.")
else:
    print(f"Befund: Der Server verarbeitet parallel. Die Gesamtrate steigt um "
          f"{beschleunigung - 1:+.0%}.\nDann ist die nächste Frage, ab welcher "
          f"Parallelität sie aufhört zu steigen.")


## 6 · Continuous Batching

Ein Scheduler kann aktive Anfragen gemeinsam durch Decoding-Schritte führen. Das erhöht den Durchsatz, kostet aber Koordination und Speicher.


In [ ]:
# ▶️ Die Ausgangszahlen: Modellgröße, Obergrenze und gemessene Rate je Modell
INSTALLIERT = {m["name"]: m for m in nativ("/api/tags")["models"]}


def groesse_gb(tag):
    """Größe des Modells, wie Ollama sie ausweist."""
    for name, m in INSTALLIERT.items():
        if name == tag or name.split(":")[0] == tag.split(":")[0]:
            return m["size"] / GB
    return None


KANDIDATEN = [MODELL, "llama3.2", "gemma4"]

RATEN = []
for tag in KANDIDATEN:
    groesse = groesse_gb(tag)
    if groesse is None:
        print(f"{tag:<14} nicht installiert, übersprungen")
        continue

    # Dieselbe Anfrage wie im Lasttest, damit die Zahlen vergleichbar bleiben.
    messe_anfrage("Warm up.", modell=tag, max_tokens=8)      # laden, zählt nicht mit
    laeufe = [messe_anfrage(LAST_PROMPT, modell=tag, max_tokens=LAST_TOKENS)
              for _ in range(5)]
    rate = statistics.median(l["tokens_pro_sekunde"] for l in laeufe)

    RATEN.append({
        "tag": tag,
        "groesse_gb": groesse,
        "obergrenze": HIER["speicherbandbreite_gb_s"] / groesse,
        "rate": rate,
    })

zeige_tabelle([{
    "Modell": m["tag"],
    "Größe (GB)": m["groesse_gb"],
    "Obergrenze (Tokens/s)": m["obergrenze"],
    "gemessen (Tokens/s)": m["rate"],
    "Ausnutzung": f"{m['rate'] / m['obergrenze']:.0%}",
} for m in RATEN])

print(f"Obergrenze = {HIER['speicherbandbreite_gb_s']} GB/s ÷ Modellgröße, eine Anfrage.")
print("Die Ausnutzung ist zugleich der Anteil der Zeit je Token, den ein Batch teilen kann:")
print("Sie ist das Verhältnis der Lesezeit zur gesamten Zeit je Token.")


In [ ]:
# Vorgegebene Hilfsfunktion
def batch_hochrechnung(rate_einzel, obergrenze, k):
    """Was k gemeinsam verarbeitete Anfragen liefern würden."""
    if rate_einzel >= obergrenze:
        raise ValueError(
            f"gemessene Rate {rate_einzel:.1f} liegt nicht unter der Obergrenze "
            f"{obergrenze:.1f} — eine der beiden Zahlen stimmt nicht")

    zeit_geteilt = 1 / obergrenze                  # Gewichte lesen, einmal je Schritt
    zeit_je_anfrage = 1 / rate_einzel - zeit_geteilt   # der Rest, je Anfrage
    zeit_schritt = zeit_geteilt + k * zeit_je_anfrage

    return {
        "k": k,
        "je_anfrage": 1 / zeit_schritt,
        "gesamt": k / zeit_schritt,
        "faktor": (k / zeit_schritt) / rate_einzel,
    }


## 7 · Kapazitätsplanung

Aus Ankunftsrate, Servicezeit und Sicherheitsreserve entsteht eine Mindestkapazität. Planung basiert auf Perzentilen, nicht auf dem Bestwert.


In [ ]:
# ▶️ Die beiden gemessenen Konstanten und das Poisson-Perzentil
# Was eine Anfrage kostet, bevor das erste Token fließt: die im Lasttest bei
# einer einzelnen Anfrage gemessene TTFT. Sie gilt für einen Prompt dieser
# Länge; für einen längeren kommt Prompt-Tokens ÷ PREFILL_RATE dazu.
TTFT_TYPISCH = basis["ttft_p50"]
RATE_EINZEL = next(m["rate"] for m in RATEN if m["tag"] == MODELL)
AUSLASTUNG_MAX = 0.70     # darüber wächst die Warteschlange schneller, als sie abgebaut wird


def poisson_p95(mittelwert):
    """Kleinste Zahl k, für die bei Poisson-Ankünften P(X ≤ k) ≥ 0,95 gilt."""
    if mittelwert <= 0:
        return 0
    wahrscheinlichkeit = math.exp(-mittelwert)
    summe, k = wahrscheinlichkeit, 0
    while summe < 0.95 and k < 1000:
        k += 1
        wahrscheinlichkeit *= mittelwert / k
        summe += wahrscheinlichkeit
    return k


print(f"TTFT je Anfrage (Abschnitt 4): {TTFT_TYPISCH:.3f} s")
print(f"Rate einer Anfrage:           {RATE_EINZEL:.0f} Tokens/s")
print(f"Zielauslastung je Replik:     {AUSLASTUNG_MAX:.0%}")
print()
for mittel in (0.1, 0.5, 1.0, 2.0, 5.0):
    print(f"   im Mittel {mittel:4.1f} gleichzeitig → in der Spitze "
          f"{poisson_p95(mittel)}")

# Gegenprobe: Trägt die Formel die Messung aus Abschnitt 4?
gerechnet = TTFT_TYPISCH + LAST_TOKENS / RATE_EINZEL
gemessen = 1 / basis["anfragen_pro_sekunde"]
print()
print(f"Gegenprobe für {LAST_TOKENS} Antwort-Tokens: Bedienzeit gerechnet "
      f"{gerechnet:.2f} s, im Lasttest bei einer Anfrage gemessen {gemessen:.2f} s "
      f"({gerechnet / gemessen - 1:+.0%})")


### 🛠️ Aufgabe 2 — Kapazität planen

Berechne aus Last, gemessener Servicezeit und Reserve, wie viele parallele Slots benötigt werden.

Führe danach den Selbsttest aus. Die ausgefüllte Variante steht in der Lösungsversion des Notebooks.


In [ ]:
def braucht_wieviel(nutzer, anfragen_pro_stunde, tokens_je_antwort, p95_ziel):
    """Prüft eine geplante Situation gegen die gemessenen Zahlen dieses Servers."""
    # TODO: Ersetze die nächste Zeile
    raise NotImplementedError("Aufgabe 2: braucht_wieviel() implementieren")


In [ ]:
# ✅ Selbsttest
FELDER = {"anfragen_je_sekunde", "bedienzeit", "kapazitaet", "auslastung", "gleichzeitig",
          "p95_geschaetzt", "erlaubt", "traegt", "repliken", "bemerkung"}

klein = braucht_wieviel(nutzer=2, anfragen_pro_stunde=10, tokens_je_antwort=200,
                        p95_ziel=30.0)
assert set(klein) == FELDER, f"Andere Schlüssel als erwartet: {set(klein) ^ FELDER}"
assert abs(klein["anfragen_je_sekunde"] - 20 / 3600) < 1e-9
assert abs(klein["bedienzeit"] - (TTFT_TYPISCH + 200 / RATE_EINZEL)) < 1e-9
assert abs(klein["kapazitaet"] * klein["bedienzeit"] - 1) < 1e-9
assert klein["traegt"] is True and klein["repliken"] == 1, klein["bemerkung"]
assert klein["auslastung"] < 0.1, "Zwei Nutzer mit zehn Anfragen je Stunde ist fast nichts"

# Mehr Last als eine Replik schafft: die Auslastung geht über 100 %.
viel = braucht_wieviel(nutzer=200, anfragen_pro_stunde=60, tokens_je_antwort=200,
                       p95_ziel=60.0)
assert viel["auslastung"] > 1.0, f"Auslastung {viel['auslastung']:.0%}"
assert viel["traegt"] is False and viel["repliken"] > 1, viel["bemerkung"]

# Antwort so lang, dass schon eine einzelne Anfrage das Ziel reißt.
lang = braucht_wieviel(nutzer=1, anfragen_pro_stunde=1, tokens_je_antwort=5000, p95_ziel=1.0)
assert lang["erlaubt"] == 0 and lang["repliken"] is None and lang["traegt"] is False, \
    "Ein einzelner Aufruf über dem Zeitziel ist kein Kapazitätsproblem"

# Jede Bemerkung ist ein Satz, keine leere Zeichenkette.
for fall in (klein, viel, lang):
    assert isinstance(fall["bemerkung"], str) and len(fall["bemerkung"]) > 20

print("✅ Aufgabe 2 gelöst")
for name, fall in (("klein", klein), ("viel", viel), ("lang", lang)):
    print(f"   {name:<6} {fall['bemerkung']}")


In [ ]:
# ▶️ Die drei Situationen aus daten/04_lasttest.json
for s in SITUATIONEN:
    print(f"{s['name']}: {s['beschreibung']}")
print()

urteile = [(s, braucht_wieviel(s["nutzer"], s["anfragen_pro_stunde"],
                               s["tokens_je_antwort"], s["p95_ziel_s"]))
           for s in SITUATIONEN]

zeige_tabelle([{
    "Situation": s["name"],
    "Nutzer": s["nutzer"],
    "Anfragen/h je Nutzer": s["anfragen_pro_stunde"],
    "Tokens je Antwort": s["tokens_je_antwort"],
    "Ziel p95 (s)": s["p95_ziel_s"],
    "Anfragen/s": u["anfragen_je_sekunde"],
    "Bedienzeit (s)": u["bedienzeit"],
    "Auslastung": f"{u['auslastung']:.0%}",
    "p95 geschätzt (s)": u["p95_geschaetzt"],
    "Repliken": u["repliken"] if u["repliken"] else "—",
} for s, u in urteile])

for s, u in urteile:
    zeichen = "✔" if u["traegt"] else "✗"
    print(f"{zeichen} {s['name']}: {u['bemerkung']}")


In [ ]:
# ▶️ Gegenprobe: Was ändert sich, wenn nur eine Zahl anders ist?
portal = next(s for s in SITUATIONEN if s["id"] == "portal")

varianten = [
    ("wie geplant", portal["nutzer"], portal["tokens_je_antwort"], portal["p95_ziel_s"]),
    ("Antwort auf 300 Tokens gekürzt", portal["nutzer"], 300, portal["p95_ziel_s"]),
    ("Zusage auf das erste Token statt auf die ganze Antwort", portal["nutzer"], 1, 1.0),
    ("300 Tokens, aber zwanzigmal so viele Nutzende", portal["nutzer"] * 20, 300,
     portal["p95_ziel_s"]),
]

geprueft = [(name, nutzer, tokens, ziel,
             braucht_wieviel(nutzer, portal["anfragen_pro_stunde"], tokens, ziel))
            for name, nutzer, tokens, ziel in varianten]

zeige_tabelle([{
    "Variante": name,
    "Nutzer": nutzer,
    "Tokens je Antwort": tokens,
    "Ziel p95 (s)": ziel,
    "Bedienzeit (s)": u["bedienzeit"],
    "Auslastung": f"{u['auslastung']:.0%}",
    "p95 geschätzt (s)": u["p95_geschaetzt"],
    "trägt": "ja" if u["traegt"] else "nein",
    "Repliken": u["repliken"] if u["repliken"] else "—",
} for name, nutzer, tokens, ziel, u in geprueft])

for nummer, (name, _, _, _, u) in enumerate(geprueft, start=1):
    print(f"{nummer} · {name}: {u['bemerkung']}")
print()
print("Zeile 3 ist kein Rechentrick: Wer die Antwort streamt, sagt die Zeit bis zum ersten")
print("Token zu, nicht bis zum letzten. In der Rechnung steht dann tokens_je_antwort=1.")


## Fazit

Du kannst Latenz und Durchsatz getrennt messen und aus Lastdaten eine Kapazitätsentscheidung ableiten. Merksatz: **Plane für Spitzen und Perzentile, nicht für den Durchschnitt.**
